In [ ]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [ ]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [ ]:
tabla_cliente = pd.read_sql_table('cliente', mensajeria)
tabla_cliente.groupby('cliente_id')
tabla_cliente.tail(10)

,cliente_id,nit_cliente,nombre,email,direccion,telefono,nombre_contacto,ciudad_id,tipo_cliente_id,activo,coordinador_id,sector
17,17,78948,CLINICA LOS CONDORES,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
18,18,9051024,UNIDAD MEDICA ESTELAR,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,S
19,20,8000569,AUTOS-RAPIDOS,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,8,1,True,NaN,industrial
20,21,900248,SALUD DE OCCIDENTE,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,S
21,22,8999020,FUNDACION CLINICA INFANTIL LOS GRITONES,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,S
22,23,900603,CLINICA MEDICA S.A.S,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
23,24,9009754,CLINICA SAN RAFAEL DE OCCIDENTE,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
24,25,900361,CALI SULUD Y VIDA,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
25,26,25478,OCCIDENTE DE SALUD-CALI,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
26,27,201478,MEDICOS Y SALUD,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,industrial


In [ ]:
dim_proveedor = tabla_cliente
dim_proveedor.rename(columns={'cliente_id':'id_proveedor','nombre':'nombre_proveedor'}, inplace=True)
#Se eliminan datos de la tabla clientes
dim_proveedor.drop(columns=['email','telefono','direccion','nombre_contacto','ciudad_id','tipo_cliente_id','activo','coordinador_id'], inplace=True)
#se eliminan los datos de la tabla sede
dim_proveedor.rename(columns={'nombre':'nombre_proveedor'},inplace=True)
dim_proveedor['key_proveedor']=range(1,len(dim_proveedor)+1)
dim_proveedor['sector']=dim_proveedor['sector'].replace('S','salud')
dim_proveedor.head()

,id_proveedor,nit_cliente,nombre_proveedor,sector,key_proveedor
0,1,25,Cliente 2,salud,1
1,2,123,Cliente 1,industrial,2
2,6,24390-3,CLINICA DEPORTIVA DEL SUR,salud,3
3,19,8301821,HOSPITAL ORTOPEDICO DE COLOMBIA,salud,4
4,8,5017350-8,CLINICA NEFROLOGOS DE CALI,salud,5


In [ ]:
dim_proveedor.to_sql('dim_proveedor', etl_conn, if_exists='replace', index=False)

27